In [1]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

In [2]:
db_path = 'krankenkasse_poc_v2.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Core-Tabelle (Statische Daten)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sap_core_versicherte (
        id INTEGER PRIMARY KEY,
        "alter" INTEGER
    )
''')

# Schatten-Tabelle (Veränderliche Daten mit Historie)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sap_shadow_historie (
        historien_id INTEGER PRIMARY KEY AUTOINCREMENT,
        id INTEGER,
        diagnose_a INTEGER,
        diagnose_b INTEGER,
        krankenhausaufenthalt INTEGER,
        krankheit_d INTEGER,
        valid_from DATE,
        valid_to DATE
    )
''')
conn.commit()

In [3]:
def fetch_historical_state(as_of_date: str) -> pd.DataFrame:
    """Holt den Datenstand zu einem bestimmten Stichtag."""
    query = f"""
        SELECT 
            c.id,
            c."alter",
            s.diagnose_a,
            s.diagnose_b,
            s.krankenhausaufenthalt,
            s.krankheit_d
        FROM sap_core_versicherte c
        JOIN sap_shadow_historie s 
          ON c.id = s.id
        WHERE '{as_of_date}' BETWEEN s.valid_from AND s.valid_to
    """
    return pd.read_sql_query(query, conn)

In [6]:
def neuen_versicherten_anlegen(v_id: int, alter: int, diag_a: int, diag_b: int, kh_aufenthalt: int, kh_d: int, gueltig_ab_str: str):
    """Legt eine neue ID im System an."""
    cursor.execute('''
        INSERT INTO sap_core_versicherte (id, "alter") 
        VALUES (?, ?)
    ''', (v_id, alter))
    
    cursor.execute('''
        INSERT INTO sap_shadow_historie 
        (id, diagnose_a, diagnose_b, krankenhausaufenthalt, krankheit_d, valid_from, valid_to) 
        VALUES (?, ?, ?, ?, ?, ?, '9999-12-31')
    ''', (v_id, diag_a, diag_b, kh_aufenthalt, kh_d, gueltig_ab_str))
    
    conn.commit()

neuen_versicherten_anlegen(
    v_id=1003, 
    alter=30, 
    diag_a=1, 
    diag_b=1, 
    kh_aufenthalt=1, 
    kh_d=1, 
    gueltig_ab_str='2026-05-21'
)

In [9]:
def update_versicherten_daten(v_id: int, diag_a: int, diag_b: int, kh_aufenthalt: int, kh_d: int, gueltig_ab_str: str):
    """Schließt den alten Zustand und öffnet einen neuen (SCD2)."""
    gueltig_ab_date = datetime.strptime(gueltig_ab_str, '%Y-%m-%d')
    altes_valid_to_str = (gueltig_ab_date - timedelta(days=1)).strftime('%Y-%m-%d')
    
    # Alten Datensatz schließen
    cursor.execute('''
        UPDATE sap_shadow_historie 
        SET valid_to = ? 
        WHERE id = ? AND valid_to = '9999-12-31'
    ''', (altes_valid_to_str, v_id))
    
    # Neuen Datensatz anlegen
    cursor.execute('''
        INSERT INTO sap_shadow_historie 
        (id, diagnose_a, diagnose_b, krankenhausaufenthalt, krankheit_d, valid_from, valid_to) 
        VALUES (?, ?, ?, ?, ?, ?, '9999-12-31')
    ''', (v_id, diag_a, diag_b, kh_aufenthalt, kh_d, gueltig_ab_str))
    
    conn.commit()
    
update_versicherten_daten(
    v_id=1001,
    diag_a=1, 
    diag_b=1, 
    kh_aufenthalt=1, 
    kh_d=1, 
    gueltig_ab_str='2026-06-01'
)

In [10]:
def export_to_csv():
    """Exportiert Tabellen als CSV."""
    pd.read_sql_query("SELECT * FROM sap_core_versicherte", conn).to_csv('export_core_v2.csv', index=False)
    pd.read_sql_query("SELECT * FROM sap_shadow_historie", conn).to_csv('export_shadow_v2.csv', index=False)

In [11]:
cursor.execute("SELECT COUNT(*) FROM sap_core_versicherte")
if cursor.fetchone()[0] == 0:
    # Patient 1: Hat anfangs nur Diagnose A
    neuen_versicherten_anlegen(v_id=1, alter=45, diag_a=1, diag_b=0, kh_aufenthalt=0, kh_d=0, gueltig_ab_str='2023-01-01')
    
    # Patient 2: Hat direkt Diagnose A, B und Krankenhausaufenthalt
    neuen_versicherten_anlegen(v_id=2, alter=62, diag_a=1, diag_b=1, kh_aufenthalt=1, kh_d=0, gueltig_ab_str='2023-05-15')

In [12]:
print("--- Stand: 2023-12-01 ---")
print(fetch_historical_state('2023-12-01').to_string(index=False))
print("\n")


--- Stand: 2023-12-01 ---
 id  alter  diagnose_a  diagnose_b  krankenhausaufenthalt  krankheit_d
  1     45           1           0                      0            0
  2     62           1           1                      1            0




In [13]:
update_versicherten_daten(v_id=1, diag_a=1, diag_b=0, kh_aufenthalt=1, kh_d=1, gueltig_ab_str='2024-02-10')

print("--- Stand Heute (Aktuell nach Update) ---")
print(fetch_historical_state('2026-05-21').to_string(index=False))
print("\n")

print("--- Kontrolle der rohen Schatten-Tabelle für Patient 1 ---")
print(pd.read_sql_query("SELECT * FROM sap_shadow_historie WHERE id = 1", conn).to_string(index=False))

--- Stand Heute (Aktuell nach Update) ---
  id  alter  diagnose_a  diagnose_b  krankenhausaufenthalt  krankheit_d
   2     62           1           1                      1            0
1003     30           1           1                      1            1
   1     45           1           0                      1            1


--- Kontrolle der rohen Schatten-Tabelle für Patient 1 ---
 historien_id  id  diagnose_a  diagnose_b  krankenhausaufenthalt  krankheit_d valid_from   valid_to
            1   1           1           0                      1            1 2024-02-10 2024-02-09
            2   1           1           0                      0            0 2023-01-01 2024-02-09
            4   1           1           0                      1            1 2024-02-10 2024-02-09
            7   1           1           0                      1            1 2024-02-10 9999-12-31


In [ ]:
conn.close() # Erst am Ende des Notebooks aufrufen